# Preprocessed Data Exploration Notebook

This notebook performs exploratory data analysis on preprocessed data for the MLOps pipeline.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import os
from glob import glob
import matplotlib.pyplot as plt
import numpy as np

# Add src directory to path
src_path = Path.cwd().parent
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from utils.config_load import Config

## Import Libraries

## Load Configuration

Load the project configuration settings from config.json

In [ ]:
config = Config.load()
print(f"Configuration loaded successfully!")
print(f"MLflow Tracking URI: {config.MLFLOW['MLFLOW_TRACKING_URI']}")
print(f"Experiment Name: {config.MLFLOW['MLFLOW_EXPERIMENT_NAME']}")
print(f"Random Seed: {config.RANDOM_SEED}")

## Load Preprocessed Data

Load CSV data from the configured preprocessed data path

In [ ]:
# Get the preprocessed data path from config
processed_data_path = config.DATA['PROCESSED_PATH']
print(f"Preprocessed data path: {processed_data_path}")

# Find all CSV files in the preprocessed data directory
if os.path.exists(processed_data_path):
    csv_files = glob(os.path.join(processed_data_path, "*.csv"))
    
    if csv_files:
        print(f"\nFound {len(csv_files)} CSV file(s):")
        for f in csv_files:
            print(f"  - {os.path.basename(f)}")
        
        # Load the first CSV file
        preprocessed_file = csv_files[0]
        print(f"\nLoading data from: {os.path.basename(preprocessed_file)}")
        df = pd.read_csv(preprocessed_file)
        print(f"✓ Data loaded successfully!")
    else:
        print("\nNo CSV files found in the preprocessed data directory.")
        print(f"Please ensure preprocessed data exists at: {processed_data_path}")
        df = None
else:
    print(f"\nDirectory not found: {processed_data_path}")
    print("Please run the preprocessing pipeline first.")
    df = None

## Initial Data Inspection

Explore the basic structure and properties of the preprocessed dataset

In [ ]:
if df is not None:
    # Data shape
    print(f"Data shape: {df.shape[0]:,} rows × {df.shape[1]} columns\n")
    
    # Column information
    print("DataFrame Info:")
    print("-" * 50)
    df.info()
    
    print("\n" + "=" * 50)
    print("First 5 rows:")
    print("=" * 50)
    display(df.head())
else:
    print("No data loaded. Please check the file path.")

## Statistical Summary

Display descriptive statistics for the preprocessed data

In [ ]:
if df is not None:
    print("Descriptive Statistics:")
    print("=" * 50)
    display(df.describe())
    
    print("\n" + "=" * 50)
    print("Missing Values:")
    print("=" * 50)
    missing = df.isnull().sum()
    missing_pct = (missing / len(df)) * 100
    missing_df = pd.DataFrame({
        'Missing Count': missing,
        'Percentage': missing_pct
    })
    display(missing_df[missing_df['Missing Count'] > 0])
    
    if missing_df['Missing Count'].sum() == 0:
        print("✓ No missing values found!")
else:
    print("No data loaded.")

## Correlation Heatmap

Visualize correlations between numeric features in the preprocessed data

In [ ]:
if df is not None:
    # Select only numeric columns
    numeric_df = df.select_dtypes(include=[np.number])
    
    if not numeric_df.empty:
        # Calculate correlation matrix
        corr_matrix = numeric_df.corr()
        
        # Create figure and axis
        fig, ax = plt.subplots(figsize=(12, 10))
        
        # Create heatmap
        im = ax.imshow(corr_matrix, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)
        
        # Set ticks and labels
        ax.set_xticks(np.arange(len(corr_matrix.columns)))
        ax.set_yticks(np.arange(len(corr_matrix.columns)))
        ax.set_xticklabels(corr_matrix.columns, rotation=45, ha='right')
        ax.set_yticklabels(corr_matrix.columns)
        
        # Add colorbar
        cbar = plt.colorbar(im, ax=ax)
        cbar.set_label('Correlation', rotation=270, labelpad=20)
        
        # Add correlation values as text
        for i in range(len(corr_matrix.columns)):
            for j in range(len(corr_matrix.columns)):
                text = ax.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                             ha='center', va='center', color='black', fontsize=8)
        
        plt.title('Preprocessed Feature Correlation Heatmap', fontsize=16, pad=20)
        plt.tight_layout()
        plt.show()
        
        print(f"\nCorrelation matrix shape: {corr_matrix.shape}")
    else:
        print("No numeric columns found in the dataset.")
else:
    print("No data loaded. Please check the file path.")